# BiLSTM Evaluation

Evaluate the baseline and fine-tuned BiLSTM checkpoints on the held-out test set.

In [13]:
import json
import sys
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, average_precision_score, classification_report, confusion_matrix, f1_score, roc_auc_score


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        marker = candidate / "distraction_prediction" / "models" / "lstm_model.py"
        if marker.exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from distraction_prediction.models.lstm_model import DistractionLSTM

WINDOWS_DIR = PROJECT_ROOT / "distraction_prediction" / "data" / "processed" / "windows"
SAVE_DIR = PROJECT_ROOT / "distraction_prediction" / "models" / "saved_models"
EVAL_DIR = SAVE_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [14]:
X_test = np.load(WINDOWS_DIR / "X_test.npy")
y_test = np.load(WINDOWS_DIR / "y_test.npy")
print(f"Test samples: {len(y_test)}")

Test samples: 6422


In [15]:
def load_checkpoint_model(checkpoint_path: Path, device: torch.device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = dict(checkpoint["config"])
    state_dict = checkpoint["model_state_dict"]
    if "head_name" not in config:
        config["head_name"] = "classifier" if any(key.startswith("classifier.") for key in state_dict) else "out"

    model = DistractionLSTM(
        input_size=config["input_size"],
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        bidirectional=config.get("bidirectional", True),
        attention_activation=config.get("attention_activation", "tanh"),
        head_name=config.get("head_name", "out"),
    ).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, config


def evaluate_checkpoint(checkpoint_path: Path, result_path: Path):
    model, config = load_checkpoint_model(checkpoint_path, device)
    expected_features = int(config["input_size"])
    if X_test.shape[2] == expected_features:
        x_eval = X_test
        compatibility_note = "exact feature match"
    elif X_test.shape[2] > expected_features:
        x_eval = X_test[:, :, :expected_features]
        compatibility_note = f"truncated test windows from {X_test.shape[2]} to {expected_features} features to match checkpoint"
    else:
        raise ValueError(
            f"Test set has {X_test.shape[2]} features but checkpoint expects {expected_features}."
        )

    with torch.no_grad():
        logits = model(torch.FloatTensor(x_eval).to(device)).squeeze()
        probabilities = torch.sigmoid(logits).cpu().numpy()

    predictions = (probabilities >= 0.5).astype(int)
    matrix = confusion_matrix(y_test, predictions)
    tn, fp, fn, tp = matrix.ravel()

    metrics = {
        "accuracy": round(float(accuracy_score(y_test, predictions)), 4),
        "f1": round(float(f1_score(y_test, predictions)), 4),
        "auc": round(float(roc_auc_score(y_test, probabilities)), 4),
        "avg_precision": round(float(average_precision_score(y_test, probabilities)), 4),
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        "config": config,
        "used_feature_count": expected_features,
        "compatibility_note": compatibility_note,
    }

    with open(result_path, "w", encoding="utf-8") as handle:
        json.dump(metrics, handle, indent=2)

    print(f"\nCheckpoint: {checkpoint_path.name}")
    print(metrics)
    print(classification_report(y_test, predictions, target_names=["Focused", "Distracted"]))

    return metrics


In [16]:
base_metrics = evaluate_checkpoint(
    SAVE_DIR / "best_model.pt",
    EVAL_DIR / "test_results.json",
)

finetuned_metrics = evaluate_checkpoint(
    SAVE_DIR / "best_model_finetuned.pt",
    EVAL_DIR / "finetuned_bilstm_results.json",
)

comparison = [
    ("BiLSTM (base)", base_metrics["accuracy"], base_metrics["f1"], base_metrics["auc"]),
    ("BiLSTM (fine-tuned)", finetuned_metrics["accuracy"], finetuned_metrics["f1"], finetuned_metrics["auc"]),
]

print("\nModel comparison")
for name, accuracy, f1, auc in comparison:
    print(f"{name:<22} accuracy={accuracy:.4f}  f1={f1:.4f}  auc={auc:.4f}")



Checkpoint: best_model.pt
{'accuracy': 0.8818, 'f1': 0.8976, 'auc': 0.9527, 'avg_precision': 0.9658, 'confusion_matrix': {'tn': 2336, 'fp': 401, 'fn': 358, 'tp': 3327}, 'config': {'input_size': 24, 'hidden_size': 64, 'num_layers': 1, 'dropout': 0.5, 'bidirectional': True, 'attention_activation': 'tanh', 'head_name': 'out'}, 'used_feature_count': 24, 'compatibility_note': 'exact feature match'}
              precision    recall  f1-score   support

     Focused       0.87      0.85      0.86      2737
  Distracted       0.89      0.90      0.90      3685

    accuracy                           0.88      6422
   macro avg       0.88      0.88      0.88      6422
weighted avg       0.88      0.88      0.88      6422


Checkpoint: best_model_finetuned.pt
{'accuracy': 0.8845, 'f1': 0.9004, 'auc': 0.9544, 'avg_precision': 0.9666, 'confusion_matrix': {'tn': 2325, 'fp': 412, 'fn': 330, 'tp': 3355}, 'config': {'input_size': 24, 'hidden_size': 64, 'num_layers': 1, 'dropout': 0.5, 'bidirectional